# Walk-Forward Time-Series Momentum

Recalibrates momentum parameters in successive walk-forward windows.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
==============================================================================
TSMOM WALK-FORWARD ANALYSIS WITH EXPANDING WINDOW - COMPLETE VERSION
==============================================================================

Walk-Forward Methodology:
-------------------------
This implementation uses an EXPANDING WINDOW approach, which is optimal for
TSMOM strategies as it incorporates all available historical information while
progressively testing on out-of-sample data.

Key Characteristics:
- Training window EXPANDS from initial size to maximum size
- Each test period uses parameters optimized only on preceding data
- No look-ahead bias: optimization at time T uses only data up to T
- Simulates realistic strategy deployment where you retrain periodically

Structure:
----------
Period 1: Train[Year 1] → Test[Year 2]
Period 2: Train[Year 1-2] → Test[Year 3]
Period 3: Train[Year 1-3] → Test[Year 4]
Period 4: Train[Year 1-4] → Test[Year 5]

Optimization Process:
---------------------
For each walk-forward period, we optimize:
1. EMA_LOOKBACK (momentum signal period)
2. VOL_LOOKBACK (volatility estimation window)
3. VOLATILITY_TARGET (target annual volatility)

Optimization metric: Sharpe Ratio on in-sample (training) data
Evaluation metric: Multiple metrics on out-of-sample (test) data
"""

from _market_data import download_price_records
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from itertools import product
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")
INITIAL_TRAIN_YEARS = 1
TEST_PERIOD_YEARS = 1
TOTAL_YEARS = 5
EMA_LOOKBACK_RANGE = [21, 42, 63, 126, 252]
VOL_LOOKBACK_RANGE = [10, 21, 42, 63]
VOLATILITY_TARGET_RANGE = [0.05, 0.1, 0.15, 0.2]
INITIAL_CAPITAL = 100000
COMPOUNDING_MODE = False
LEVERAGE = 1.0
REBALANCE_FREQUENCY = "M"
COMMISSION_BPS = 5.0
SLIPPAGE_BPS = 2.0
TOTAL_COST_BPS = COMMISSION_BPS + SLIPPAGE_BPS
START_DATE = "2019-12-31"
END_DATE = "2024-12-31"
BENCHMARK_TICKER = "URTH"
UNIVERSE = ["SPY", "QQQ", "IWM", "EFA", "EEM", "AGG", "TLT", "GLD", "DBC", "VNQ"]


def fetch_yahoo_data(ticker, start_date, end_date):
    """Fetch historical OHLCV data from Yahoo Finance."""
    try:
        response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
        data = response_prices
        if "historical" not in data or len(data["historical"]) == 0:
            return None
        df = pd.DataFrame(data["historical"])
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values("date").set_index("date")
        df = df.rename(
            columns={
                "open": "Open",
                "high": "High",
                "low": "Low",
                "close": "Close",
                "volume": "Volume",
            }
        )
        return df[["Open", "High", "Low", "Close", "Volume"]]
    except Exception as e:
        print(f"ERROR fetching {ticker}: {str(e)}")
        return None


def fetch_universe_data(tickers, start_date, end_date):
    """Fetch data for entire universe."""
    data_dict = {}
    print(f"\nFetching data for {len(tickers)} tickers...")
    for ticker in tickers:
        df = fetch_yahoo_data(ticker, start_date, end_date)
        if df is not None and len(df) > 0:
            data_dict[ticker] = df
            print(f"  ✓ {ticker}: {len(df)} bars")
    return data_dict


def calculate_ema_returns(prices, lookback):
    """Calculate EMA of log returns."""
    log_returns = np.log(prices / prices.shift(1))
    return log_returns.ewm(span=lookback, adjust=False).mean()


def calculate_rolling_volatility(prices, lookback):
    """Calculate annualized rolling volatility."""
    log_returns = np.log(prices / prices.shift(1))
    rolling_vol = log_returns.rolling(window=lookback).std()
    annualized_vol = rolling_vol * np.sqrt(252)
    return annualized_vol.clip(lower=0.01)


def generate_tsmom_signals(prices, ema_lookback, vol_lookback, vol_target):
    """Generate TSMOM signals with volatility scaling."""
    ema_returns = calculate_ema_returns(prices, ema_lookback)
    momentum_signal = np.sign(ema_returns)
    realized_vol = calculate_rolling_volatility(prices, vol_lookback)
    vol_scalar = (vol_target / realized_vol).clip(upper=2.0)
    position_size = momentum_signal * vol_scalar
    return pd.DataFrame(
        {
            "momentum_signal": momentum_signal,
            "realized_vol": realized_vol,
            "vol_scalar": vol_scalar,
            "position_size": position_size,
        },
        index=prices.index,
    )


def get_rebalance_dates(price_index, frequency="M"):
    """Get rebalancing dates."""
    if frequency == "M":
        return price_index.to_series().resample("ME").last().index.tolist()
    elif frequency == "W":
        return price_index.to_series().resample("W").last().index.tolist()
    else:
        return price_index.tolist()


def calculate_portfolio_returns(
    data_dict, signals_dict, rebalance_dates, initial_capital, compounding_mode, leverage, cost_bps
):
    """
    Calculate portfolio returns with proper mark-to-market P&L tracking.
    This is the CORRECTED version that properly accumulates portfolio value.
    """
    common_dates = None
    for ticker, df in data_dict.items():
        if common_dates is None:
            common_dates = df.index
        else:
            common_dates = common_dates.intersection(df.index)
    common_dates = sorted(common_dates)
    portfolio_value = initial_capital
    capital_base = initial_capital
    positions = {}
    pending_signals = {}
    results = []
    for i, date in enumerate(common_dates):
        if date in rebalance_dates:
            pending_signals = {}
            for ticker in data_dict.keys():
                if date not in signals_dict[ticker].index:
                    continue
                target_position = signals_dict[ticker].loc[date, "position_size"]
                if pd.notna(target_position):
                    pending_signals[ticker] = target_position
        daily_pnl = 0.0
        if len(positions) > 0:
            for ticker, pos_info in positions.items():
                if date not in data_dict[ticker].index:
                    continue
                shares = pos_info["shares"]
                prev_close = pos_info["prev_close"]
                current_close = data_dict[ticker].loc[date, "Close"]
                if pd.isna(current_close) or current_close <= 0:
                    continue
                pnl = (current_close - prev_close) * shares
                daily_pnl += pnl
                positions[ticker]["prev_close"] = current_close
        portfolio_value += daily_pnl
        transaction_costs = 0.0
        if i > 0 and len(pending_signals) > 0:
            sizing_capital = portfolio_value if compounding_mode else capital_base
            num_assets = len(pending_signals)
            capital_per_asset = sizing_capital * leverage / num_assets if num_assets > 0 else 0
            new_positions = {}
            for ticker, target_signal in pending_signals.items():
                if date not in data_dict[ticker].index:
                    continue
                open_price = data_dict[ticker].loc[date, "Open"]
                close_price = data_dict[ticker].loc[date, "Close"]
                if pd.isna(open_price) or open_price <= 0:
                    continue
                if pd.isna(close_price) or close_price <= 0:
                    close_price = open_price
                target_exposure = capital_per_asset * target_signal
                current_exposure = 0
                if ticker in positions:
                    current_exposure = positions[ticker]["shares"] * open_price
                exposure_change = target_exposure - current_exposure
                trade_cost = abs(exposure_change) * (cost_bps / 10000)
                transaction_costs += trade_cost
                if abs(target_exposure) > 1:
                    shares = target_exposure / open_price
                    new_positions[ticker] = {
                        "shares": shares,
                        "entry_price": open_price,
                        "prev_close": close_price,
                    }
            positions = new_positions
            portfolio_value -= transaction_costs
            pending_signals = {}
        if i == 0:
            daily_return = 0.0
        else:
            previous_value = results[-1]["portfolio_value"]
            daily_return = portfolio_value / previous_value - 1 if previous_value > 0 else 0.0
        gross_exposure = sum((abs(p["shares"] * p["prev_close"]) for p in positions.values()))
        results.append(
            {
                "date": date,
                "portfolio_value": portfolio_value,
                "daily_return": daily_return,
                "num_positions": len(positions),
                "gross_exposure": gross_exposure,
            }
        )
    return pd.DataFrame(results).set_index("date")


def calculate_sharpe_ratio(returns_series):
    """Calculate annualized Sharpe ratio."""
    if len(returns_series) == 0 or returns_series.std() == 0:
        return 0.0
    return returns_series.mean() * 252 / (returns_series.std() * np.sqrt(252))


def calculate_performance_metrics(returns_series):
    """Calculate comprehensive performance metrics."""
    if len(returns_series) == 0:
        return {}
    returns = returns_series.dropna()
    total_return = (1 + returns).prod() - 1
    years = len(returns) / 252
    annualized_return = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0
    volatility = returns.std() * np.sqrt(252)
    sharpe = annualized_return / volatility if volatility > 0 else 0
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min()
    calmar = abs(annualized_return / max_drawdown) if max_drawdown != 0 else 0
    win_rate = (returns > 0).sum() / len(returns) if len(returns) > 0 else 0
    downside_returns = returns[returns < 0]
    downside_dev = downside_returns.std() * np.sqrt(252)
    sortino = annualized_return / downside_dev if downside_dev > 0 else 0
    return {
        "Total Return": total_return,
        "Annualized Return": annualized_return,
        "Annualized Volatility": volatility,
        "Sharpe Ratio": sharpe,
        "Sortino Ratio": sortino,
        "Maximum Drawdown": max_drawdown,
        "Calmar Ratio": calmar,
        "Win Rate": win_rate,
    }


def optimize_parameters(data_dict, train_start, train_end):
    """
    Optimize TSMOM parameters on training data.

    Returns best parameters based on in-sample Sharpe ratio.

    CRITICAL: Only uses data from train_start to train_end (no look-ahead).
    """
    print(f"\n  Optimizing on training period: {train_start.date()} to {train_end.date()}")
    train_data = {}
    for ticker, df in data_dict.items():
        train_data[ticker] = df.loc[train_start:train_end]
    common_dates = None
    for df in train_data.values():
        if common_dates is None:
            common_dates = df.index
        else:
            common_dates = common_dates.intersection(df.index)
    common_dates = sorted(common_dates)
    if len(common_dates) < 100:
        print(f"  WARNING: Insufficient training data ({len(common_dates)} days)")
        return None
    rebalance_dates = get_rebalance_dates(pd.DatetimeIndex(common_dates), REBALANCE_FREQUENCY)
    best_sharpe = -999
    best_params = None
    total_combinations = (
        len(EMA_LOOKBACK_RANGE) * len(VOL_LOOKBACK_RANGE) * len(VOLATILITY_TARGET_RANGE)
    )
    print(f"  Testing {total_combinations} parameter combinations...")
    combination_count = 0
    for ema_lb in EMA_LOOKBACK_RANGE:
        for vol_lb in VOL_LOOKBACK_RANGE:
            for vol_target in VOLATILITY_TARGET_RANGE:
                combination_count += 1
                signals_dict = {}
                for ticker, df in train_data.items():
                    signals = generate_tsmom_signals(
                        prices=df["Close"],
                        ema_lookback=ema_lb,
                        vol_lookback=vol_lb,
                        vol_target=vol_target,
                    )
                    signals_dict[ticker] = signals
                try:
                    results = calculate_portfolio_returns(
                        data_dict=train_data,
                        signals_dict=signals_dict,
                        rebalance_dates=rebalance_dates,
                        initial_capital=INITIAL_CAPITAL,
                        compounding_mode=COMPOUNDING_MODE,
                        leverage=LEVERAGE,
                        cost_bps=TOTAL_COST_BPS,
                    )
                    sharpe = calculate_sharpe_ratio(results["daily_return"])
                    if sharpe > best_sharpe:
                        best_sharpe = sharpe
                        best_params = {
                            "ema_lookback": ema_lb,
                            "vol_lookback": vol_lb,
                            "volatility_target": vol_target,
                            "in_sample_sharpe": sharpe,
                        }
                except Exception as e:
                    continue
    if best_params is None:
        print("  ERROR: No valid parameter combinations found")
        return None
    print(
        f"  ✓ Best parameters: EMA={best_params['ema_lookback']}, Vol={best_params['vol_lookback']}, Target={best_params['volatility_target'] * 100:.0f}%, Sharpe={best_params['in_sample_sharpe']:.2f}"
    )
    return best_params


def run_walkforward_analysis(data_dict, benchmark_data):
    """
    Execute walk-forward analysis with expanding window.

    Structure:
    - Period 1: Train on Year 1, Test on Year 2
    - Period 2: Train on Year 1-2, Test on Year 3
    - Period 3: Train on Year 1-3, Test on Year 4
    - Period 4: Train on Year 1-4, Test on Year 5
    """
    print("\n" + "=" * 80)
    print("WALK-FORWARD ANALYSIS WITH EXPANDING WINDOW")
    print("=" * 80)
    common_dates = None
    for df in data_dict.values():
        if common_dates is None:
            common_dates = df.index
        else:
            common_dates = common_dates.intersection(df.index)
    common_dates = sorted(common_dates)
    start_date = common_dates[0]
    end_date = common_dates[-1]
    print(f"Total data period: {start_date.date()} to {end_date.date()}")
    print(f"Initial training: {INITIAL_TRAIN_YEARS} year(s)")
    print(f"Test periods: {TEST_PERIOD_YEARS} year(s) each")
    print(f"Expanding window: Training size increases each period")
    periods = []
    current_train_end = start_date + pd.DateOffset(years=INITIAL_TRAIN_YEARS)
    period_num = 1
    while current_train_end < end_date:
        test_start = current_train_end + pd.DateOffset(days=1)
        test_end = test_start + pd.DateOffset(years=TEST_PERIOD_YEARS) - pd.DateOffset(days=1)
        if test_end > end_date:
            test_end = end_date
        periods.append(
            {
                "period": period_num,
                "train_start": start_date,
                "train_end": current_train_end,
                "test_start": test_start,
                "test_end": test_end,
            }
        )
        current_train_end = test_end
        period_num += 1
    print(f"\nTotal walk-forward periods: {len(periods)}")
    print("-" * 80)
    wf_results = []
    all_oos_returns = []
    for period_info in periods:
        print(f"\n{'=' * 80}")
        print(f"PERIOD {period_info['period']}")
        print(f"{'=' * 80}")
        print(f"Training: {period_info['train_start'].date()} to {period_info['train_end'].date()}")
        print(f"Testing:  {period_info['test_start'].date()} to {period_info['test_end'].date()}")
        best_params = optimize_parameters(
            data_dict=data_dict,
            train_start=period_info["train_start"],
            train_end=period_info["train_end"],
        )
        if best_params is None:
            print("  Skipping period due to optimization failure")
            continue
        print(
            f"\n  Testing on OOS period: {period_info['test_start'].date()} to {period_info['test_end'].date()}"
        )
        test_data = {}
        for ticker, df in data_dict.items():
            test_data[ticker] = df.loc[period_info["test_start"] : period_info["test_end"]]
        signals_dict = {}
        for ticker, df in test_data.items():
            signals = generate_tsmom_signals(
                prices=df["Close"],
                ema_lookback=best_params["ema_lookback"],
                vol_lookback=best_params["vol_lookback"],
                vol_target=best_params["volatility_target"],
            )
            signals_dict[ticker] = signals
        test_common_dates = None
        for df in test_data.values():
            if test_common_dates is None:
                test_common_dates = df.index
            else:
                test_common_dates = test_common_dates.intersection(df.index)
        test_common_dates = sorted(test_common_dates)
        if len(test_common_dates) < 20:
            print(f"  WARNING: Insufficient test data ({len(test_common_dates)} days)")
            continue
        test_rebalance_dates = get_rebalance_dates(
            pd.DatetimeIndex(test_common_dates), REBALANCE_FREQUENCY
        )
        test_results = calculate_portfolio_returns(
            data_dict=test_data,
            signals_dict=signals_dict,
            rebalance_dates=test_rebalance_dates,
            initial_capital=INITIAL_CAPITAL,
            compounding_mode=COMPOUNDING_MODE,
            leverage=LEVERAGE,
            cost_bps=TOTAL_COST_BPS,
        )
        oos_metrics = calculate_performance_metrics(test_results["daily_return"])
        bench_test = benchmark_data.loc[period_info["test_start"] : period_info["test_end"]]
        bench_returns = bench_test["Close"].pct_change()
        bench_metrics = calculate_performance_metrics(bench_returns)
        print(f"\n  OUT-OF-SAMPLE RESULTS:")
        print(f"    Total Return:     {oos_metrics['Total Return'] * 100:>7.2f}%")
        print(f"    Annualized:       {oos_metrics['Annualized Return'] * 100:>7.2f}%")
        print(f"    Volatility:       {oos_metrics['Annualized Volatility'] * 100:>7.2f}%")
        print(f"    Sharpe Ratio:     {oos_metrics['Sharpe Ratio']:>7.2f}")
        print(f"    Max Drawdown:     {oos_metrics['Maximum Drawdown'] * 100:>7.2f}%")
        print(f"    Win Rate:         {oos_metrics['Win Rate'] * 100:>7.2f}%")
        print(f"\n  BENCHMARK COMPARISON:")
        print(f"    Benchmark Return: {bench_metrics['Total Return'] * 100:>7.2f}%")
        print(
            f"    Outperformance:   {(oos_metrics['Total Return'] - bench_metrics['Total Return']) * 100:>7.2f}%"
        )
        wf_results.append(
            {
                "period": period_info["period"],
                "train_start": period_info["train_start"],
                "train_end": period_info["train_end"],
                "test_start": period_info["test_start"],
                "test_end": period_info["test_end"],
                "ema_lookback": best_params["ema_lookback"],
                "vol_lookback": best_params["vol_lookback"],
                "volatility_target": best_params["volatility_target"],
                "in_sample_sharpe": best_params["in_sample_sharpe"],
                "oos_total_return": oos_metrics["Total Return"],
                "oos_ann_return": oos_metrics["Annualized Return"],
                "oos_volatility": oos_metrics["Annualized Volatility"],
                "oos_sharpe": oos_metrics["Sharpe Ratio"],
                "oos_sortino": oos_metrics["Sortino Ratio"],
                "oos_max_dd": oos_metrics["Maximum Drawdown"],
                "oos_calmar": oos_metrics["Calmar Ratio"],
                "oos_win_rate": oos_metrics["Win Rate"],
                "bench_return": bench_metrics["Total Return"],
                "outperformance": oos_metrics["Total Return"] - bench_metrics["Total Return"],
            }
        )
        all_oos_returns.append(test_results["daily_return"])
    return (pd.DataFrame(wf_results), pd.concat(all_oos_returns))


def plot_walkforward_results(wf_df):
    """Create comprehensive walk-forward analysis visualizations."""
    fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    fig.suptitle("TSMOM Walk-Forward Analysis Results", fontsize=16, fontweight="bold")
    ax1 = axes[0, 0]
    x_pos = range(len(wf_df))
    ax1.bar(x_pos, wf_df["oos_total_return"] * 100, alpha=0.7, label="Strategy")
    ax1.bar(x_pos, wf_df["bench_return"] * 100, alpha=0.5, label="Benchmark")
    ax1.set_title("Out-of-Sample Returns by Period")
    ax1.set_xlabel("Period")
    ax1.set_ylabel("Return (%)")
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels([f"P{p}" for p in wf_df["period"]])
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis="y")
    ax1.axhline(y=0, color="black", linestyle="-", linewidth=0.8)
    ax2 = axes[0, 1]
    x_pos = range(len(wf_df))
    width = 0.35
    ax2.bar(
        [x - width / 2 for x in x_pos],
        wf_df["in_sample_sharpe"],
        width,
        label="In-Sample",
        alpha=0.7,
    )
    ax2.bar(
        [x + width / 2 for x in x_pos], wf_df["oos_sharpe"], width, label="Out-of-Sample", alpha=0.7
    )
    ax2.set_title("Sharpe Ratio: In-Sample vs Out-of-Sample")
    ax2.set_xlabel("Period")
    ax2.set_ylabel("Sharpe Ratio")
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels([f"P{p}" for p in wf_df["period"]])
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis="y")
    ax2.axhline(y=0, color="black", linestyle="-", linewidth=0.8)
    ax3 = axes[1, 0]
    ax3.plot(wf_df["period"], wf_df["ema_lookback"], marker="o", linewidth=2, markersize=8)
    ax3.set_title("Optimized EMA Lookback Period")
    ax3.set_xlabel("Period")
    ax3.set_ylabel("EMA Lookback (days)")
    ax3.grid(True, alpha=0.3)
    ax4 = axes[1, 1]
    ax4.plot(
        wf_df["period"],
        wf_df["volatility_target"] * 100,
        marker="s",
        linewidth=2,
        markersize=8,
        color="orange",
    )
    ax4.set_title("Optimized Volatility Target")
    ax4.set_xlabel("Period")
    ax4.set_ylabel("Vol Target (%)")
    ax4.grid(True, alpha=0.3)
    ax5 = axes[2, 0]
    ax5.bar(range(len(wf_df)), wf_df["oos_max_dd"] * 100, color="red", alpha=0.6)
    ax5.set_title("Out-of-Sample Maximum Drawdown")
    ax5.set_xlabel("Period")
    ax5.set_ylabel("Max Drawdown (%)")
    ax5.set_xticks(range(len(wf_df)))
    ax5.set_xticklabels([f"P{p}" for p in wf_df["period"]])
    ax5.grid(True, alpha=0.3, axis="y")
    ax6 = axes[2, 1]
    colors = ["green" if x > 0 else "red" for x in wf_df["outperformance"]]
    ax6.bar(range(len(wf_df)), wf_df["outperformance"] * 100, color=colors, alpha=0.6)
    ax6.set_title("Outperformance vs Benchmark")
    ax6.set_xlabel("Period")
    ax6.set_ylabel("Excess Return (%)")
    ax6.set_xticks(range(len(wf_df)))
    ax6.set_xticklabels([f"P{p}" for p in wf_df["period"]])
    ax6.axhline(y=0, color="black", linestyle="-", linewidth=0.8)
    ax6.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()


def print_walkforward_summary(wf_df, aggregate_oos_returns):
    """Print comprehensive walk-forward summary statistics."""
    print("\n" + "=" * 80)
    print("WALK-FORWARD ANALYSIS SUMMARY")
    print("=" * 80)
    print("\nPER-PERIOD RESULTS:")
    print("-" * 80)
    print(
        f"{'Period':<8} {'Train End':<12} {'Test End':<12} {'EMA':<6} {'Vol':<6} {'IS Sharpe':<10} {'OOS Ret%':<10} {'OOS Sharpe':<11} {'Outperf%':<10}"
    )
    print("-" * 80)
    for _, row in wf_df.iterrows():
        print(
            f"{row['period']:<8} {row['train_end'].strftime('%Y-%m-%d'):<12} {row['test_end'].strftime('%Y-%m-%d'):<12} {row['ema_lookback']:<6.0f} {row['vol_lookback']:<6.0f} {row['in_sample_sharpe']:<10.2f} {row['oos_total_return'] * 100:<10.2f} {row['oos_sharpe']:<11.2f} {row['outperformance'] * 100:<10.2f}"
        )
    print("\n" + "=" * 80)
    print("AGGREGATE OUT-OF-SAMPLE PERFORMANCE")
    print("=" * 80)
    agg_metrics = calculate_performance_metrics(aggregate_oos_returns)
    print(f"\nTotal Return (all OOS periods):    {agg_metrics['Total Return'] * 100:>8.2f}%")
    print(f"Annualized Return:                 {agg_metrics['Annualized Return'] * 100:>8.2f}%")
    print(f"Annualized Volatility:             {agg_metrics['Annualized Volatility'] * 100:>8.2f}%")
    print(f"Sharpe Ratio:                      {agg_metrics['Sharpe Ratio']:>8.2f}")
    print(f"Sortino Ratio:                     {agg_metrics['Sortino Ratio']:>8.2f}")
    print(f"Maximum Drawdown:                  {agg_metrics['Maximum Drawdown'] * 100:>8.2f}%")
    print(f"Calmar Ratio:                      {agg_metrics['Calmar Ratio']:>8.2f}")
    print(f"Win Rate:                          {agg_metrics['Win Rate'] * 100:>8.2f}%")
    print("\n" + "=" * 80)
    print("PARAMETER STABILITY ANALYSIS")
    print("=" * 80)
    ema_mean = wf_df["ema_lookback"].mean()
    ema_std = wf_df["ema_lookback"].std()
    vol_mean = wf_df["vol_lookback"].mean()
    vol_std = wf_df["vol_lookback"].std()
    target_mean = wf_df["volatility_target"].mean()
    target_std = wf_df["volatility_target"].std()
    print(f"\nEMA Lookback:        Mean = {ema_mean:.1f}, Std Dev = {ema_std:.1f}")
    print(f"Vol Lookback:        Mean = {vol_mean:.1f}, Std Dev = {vol_std:.1f}")
    print(
        f"Volatility Target:   Mean = {target_mean * 100:.1f}%, Std Dev = {target_std * 100:.1f}%"
    )
    print("\n" + "=" * 80)
    print("IN-SAMPLE vs OUT-OF-SAMPLE SHARPE COMPARISON")
    print("=" * 80)
    avg_is_sharpe = wf_df["in_sample_sharpe"].mean()
    avg_oos_sharpe = wf_df["oos_sharpe"].mean()
    sharpe_degradation = (
        (avg_is_sharpe - avg_oos_sharpe) / avg_is_sharpe * 100 if avg_is_sharpe != 0 else 0
    )
    print(f"\nAverage In-Sample Sharpe:          {avg_is_sharpe:>8.2f}")
    print(f"Average Out-of-Sample Sharpe:      {avg_oos_sharpe:>8.2f}")
    print(f"Sharpe Degradation:                {sharpe_degradation:>8.1f}%")
    if sharpe_degradation < 20:
        print("\n✓ LOW OVERFITTING RISK: OOS performance is stable")
    elif sharpe_degradation < 40:
        print("\n⚠ MODERATE OVERFITTING RISK: Some performance degradation")
    else:
        print("\n✗ HIGH OVERFITTING RISK: Significant IS/OOS discrepancy")
    print("\n" + "=" * 80)
    print("CONSISTENCY METRICS")
    print("=" * 80)
    positive_periods = (wf_df["oos_total_return"] > 0).sum()
    total_periods = len(wf_df)
    consistency_ratio = positive_periods / total_periods if total_periods > 0 else 0
    outperforming_periods = (wf_df["outperformance"] > 0).sum()
    outperformance_ratio = outperforming_periods / total_periods if total_periods > 0 else 0
    print(
        f"\nPositive Return Periods:           {positive_periods}/{total_periods} ({consistency_ratio * 100:.1f}%)"
    )
    print(
        f"Benchmark Outperformance Periods:  {outperforming_periods}/{total_periods} ({outperformance_ratio * 100:.1f}%)"
    )
    print("\n" + "=" * 80 + "\n")


def main():
    """Main execution function for walk-forward analysis."""
    print("\n" + "=" * 80)
    print("TSMOM WALK-FORWARD ANALYSIS - EXPANDING WINDOW")
    print("=" * 80)
    print(f"Backtest Period: {START_DATE} to {END_DATE}")
    print(f"Initial Capital: ${INITIAL_CAPITAL:,.0f}")
    print(
        f"Optimization Grid Size: {len(EMA_LOOKBACK_RANGE) * len(VOL_LOOKBACK_RANGE) * len(VOLATILITY_TARGET_RANGE)} combinations"
    )
    print("=" * 80)
    print("\nSTEP 1: DATA ACQUISITION")
    print("-" * 80)
    data_dict = fetch_universe_data(UNIVERSE, START_DATE, END_DATE)
    if len(data_dict) == 0:
        print("ERROR: No data fetched. Exiting.")
        return
    benchmark_data = fetch_yahoo_data(BENCHMARK_TICKER, START_DATE, END_DATE)
    if benchmark_data is None:
        print(f"WARNING: Could not fetch benchmark {BENCHMARK_TICKER}")
        return
    print("\nSTEP 2: WALK-FORWARD OPTIMIZATION AND TESTING")
    print("-" * 80)
    wf_results_df, aggregate_oos_returns = run_walkforward_analysis(data_dict, benchmark_data)
    print_walkforward_summary(wf_results_df, aggregate_oos_returns)
    print("\nSTEP 3: VISUALIZATION")
    print("-" * 80)
    plot_walkforward_results(wf_results_df)
    print("\n✓ WALK-FORWARD ANALYSIS COMPLETE\n")
    return (wf_results_df, aggregate_oos_returns)


if __name__ == "__main__":
    wf_results, oos_returns = main()
